<a href="https://colab.research.google.com/github/shaan-byte/python_ds_colab/blob/main/Managing_API_Complexity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 4.2 — Managing API Complexity

**Module 1: Programming & Data Foundations**  

---

## What we will cover today

1. Status codes — reading and handling them properly
2. Authentication — API keys and how to use them safely
3. Header management — sending and reading headers
4. Pagination — fetching large datasets one page at a time

---

---
## Setup — Run this first

In [ ]:
import requests
import json
import time

print("requests version:", requests.__version__)
print("Ready.")

requests version: 2.32.4
Ready.


---
# Part 1 — Status Codes

## A quick recap

In the last session we saw that every response comes with a **status code** — a three-digit number that tells you whether your request worked. We listed the common ones. Today we go further: how do you actually write code that handles them?

---

## Why handling status codes matters

Consider this code:

```python
response = requests.get("https://api.example.com/weather?city=mumbai")
data = response.json()
print(data["temperature"])
```

This looks fine. But what if:
- The city name was wrong — the server returns a 404
- Your API key expired — the server returns a 401
- The server is down — the server returns a 500

In all three cases, `response.json()` will either fail completely or return an error message rather than weather data. Then `data["temperature"]` will throw a `KeyError`. Your program crashes with no useful information about what went wrong.

Handling status codes makes your programs **robust** — they fail gracefully with a clear message instead of crashing with a confusing error.

---

## The full status code map

| Code | Name | What it means | Your response |
|---|---|---|---|
| `200` | OK | Everything worked | Use the data |
| `201` | Created | POST succeeded, new resource was created | Confirm creation |
| `204` | No Content | Worked, but there is nothing to return | Treat as success |
| `400` | Bad Request | Your request is malformed | Fix your code or inputs |
| `401` | Unauthorized | No credentials, or credentials are wrong | Check your API key |
| `403` | Forbidden | Credentials are valid but you lack permission | You need a higher-tier plan |
| `404` | Not Found | That resource does not exist | Check the URL or resource ID |
| `429` | Too Many Requests | You have exceeded the rate limit | Wait and retry |
| `500` | Internal Server Error | The server itself broke | Not your fault — retry later |
| `503` | Service Unavailable | Server is temporarily down | Retry later |

A useful mental model: **4xx = your fault, 5xx = their fault.**

In [ ]:
# Trigger different status codes deliberately using HTTPBin
# HTTPBin has endpoints that return any status code you ask for

codes_to_test = [200, 201, 400, 401, 403, 404, 429, 500]

print(f"{'Code':<6} {'Category':<12} {'response.ok':<14} {'Meaning'}")
print("-" * 60)

for code in codes_to_test:
    r = requests.get(f"https://httpbin.org/status/{code}")
    print(r.status_code)

    if 200 <= code < 300:
        category = "Success"
    elif 400 <= code < 500:
        category = "Client error"
    else:
        category = "Server error"

    # response.ok is True only for 2xx codes
    print(f"  {code:<4} {category:<12} {str(r.ok):<14} ok={r.ok}")

Code   Category     response.ok    Meaning
------------------------------------------------------------
502
  200  Success      False          ok=False
201
  201  Success      True           ok=True
400
  400  Client error False          ok=False
401
  401  Client error False          ok=False
403
  403  Client error False          ok=False
404
  404  Client error False          ok=False
429
  429  Client error False          ok=False
500
  500  Server error False          ok=False


### `response.ok` — the quick check

`response.ok` is a shortcut: it returns `True` if the status code is in the 200–299 range, and `False` for everything else. Use it when you just need a simple pass/fail check.

For more specific handling — treating 404 differently from 429, for example — check `response.status_code` directly.

In [ ]:
# Pattern 1: Simple check with response.ok
# Use when you just need success vs failure

def simple_fetch(url):
    response = requests.get(url)
    if response.ok:
        return response.json()
    else:
        print(f"Request failed with status {response.status_code}")
        return None

print("Pattern 1 — simple check:")
print(simple_fetch("https://restcountries.com/v3.1/name/india?fields=name"))
print(simple_fetch("https://restcountries.com/v3.1/name/zzzfake"))
print()

In [ ]:
# Pattern 2: Granular handling — treat each error category differently
# This is what real production code looks like

def fetch_with_handling(url, params=None):
    """
    Makes a GET request and handles each status code category explicitly.
    Returns the parsed JSON on success, or None on failure.
    """
    try:
        response = requests.get(url, params=params, timeout=10)
        # timeout=10 means: if the server does not respond in 10 seconds, give up

    except requests.exceptions.ConnectionError:
        print("ERROR: Could not connect to the server. Check your internet connection.")
        return None
    except requests.exceptions.Timeout:
        print("ERROR: The request timed out. The server took too long to respond.")
        return None

    code = response.status_code

    if code == 200:
        return response.json()

    elif code == 400:
        print(f"ERROR 400: Bad request. Check your parameters.")

    elif code == 401:
        print(f"ERROR 401: Unauthorized. Your API key is missing or invalid.")

    elif code == 403:
        print(f"ERROR 403: Forbidden. You do not have permission for this resource.")

    elif code == 404:
        print(f"ERROR 404: Not found. The resource at '{url}' does not exist.")

    elif code == 429:
        print(f"ERROR 429: Rate limit exceeded. Wait before trying again.")

    elif code >= 500:
        print(f"ERROR {code}: Server error. This is not your fault — try again later.")

    else:
        print(f"Unexpected status code: {code}")

    return None


# Test with a good URL
print("--- Good request ---")
result = fetch_with_handling("https://restcountries.com/v3.1/name/japan", params={"fields": "name,capital"})
if result:
    print("Success:", result[0]["name"]["common"])

print()

# Test with a bad URL
print("--- Bad request (nonexistent country) ---")
fetch_with_handling("https://restcountries.com/v3.1/name/notacountryatall")

In [ ]:
# Pattern 3: raise_for_status() — let requests throw an exception for you
# Useful when you want to catch errors in a try/except block

# response.raise_for_status() does nothing on 2xx
# and raises requests.exceptions.HTTPError on 4xx or 5xx

urls_to_try = [
    "https://restcountries.com/v3.1/name/germany?fields=name",
    "https://restcountries.com/v3.1/name/zzznotreal"
]

for url in urls_to_try:
    try:
        r = requests.get(url)
        r.raise_for_status()    # raises HTTPError if status >= 400
        data = r.json()
        print(f"OK    | {data[0]['name']['common']}")

    except requests.exceptions.HTTPError as e:
        print(f"FAIL  | HTTPError: {e}")

---
### Exercise 1 — Status Code Handling

**Task:** Complete the function below. It should call the REST Countries API for a given country name and return the country's population. If anything goes wrong, it should return `None` and print a helpful message.

Handle these cases specifically:
- Status `200` — return the population as an integer
- Status `404` — print `"Country not found"` and return `None`
- Any other status — print the code and return `None`

In [ ]:
def get_population(country_name):
    """
    Returns the population of a country, or None if not found.
    """
    url    = f"https://restcountries.com/v3.1/name/{country_name}"
    params = {"fields": "name,population"}

    response = requests.get(url, params=params)
    code     = response.___          # <-- how do you get the status code?

    if code == ___:                  # <-- success case
        data = response.json()
        return data[0][___]          # <-- which field?

    elif code == ___:                # <-- not found case
        print(f"Country not found: '{country_name}'")
        return None

    else:
        print(f"Unexpected status code ___ for '{country_name}'")   # <-- embed the code
        return None


# Test it
pop1 = get_population("brazil")
print(f"Brazil population:       {pop1:,}" if pop1 else "No result")

pop2 = get_population("fakecountryxyz")
print(f"Fake country population: {pop2}")

print()
print(f"Correct (Brazil found):       {pop1 is not None and pop1 > 200_000_000}")
print(f"Correct (fake returns None):  {pop2 is None}")

<details>
<summary>Show solution</summary>

```python
def get_population(country_name):
    url    = f"https://restcountries.com/v3.1/name/{country_name}"
    params = {"fields": "name,population"}
    response = requests.get(url, params=params)
    code     = response.status_code

    if code == 200:
        data = response.json()
        return data[0]["population"]
    elif code == 404:
        print(f"Country not found: '{country_name}'")
        return None
    else:
        print(f"Unexpected status code {code} for '{country_name}'")
        return None
```

</details>

---
# Part 2 — Authentication and API Keys

## Why do APIs need authentication?

So far, every API we used was public — no login needed. But most real-world APIs require you to identify yourself. There are three main reasons:

**1. Rate limiting** — A free API key allows 60 requests per minute. A paid key allows 1000. Without a key, the API cannot enforce these limits per user.

**2. Billing** — If you use more than the free tier, the API provider needs to know who to charge.

**3. Access control** — Some data is sensitive. A company's internal API should only respond to requests from that company's own applications.

---

## The main authentication methods

| Method | How it works | Seen in |
|---|---|---|
| **API Key (query param)** | Key is added to the URL: `?appid=abc123` | OpenWeatherMap, many simple APIs |
| **API Key (header)** | Key is sent in the request header: `X-API-Key: abc123` | More secure — key not visible in URL |
| **Bearer token** | `Authorization: Bearer <token>` header | GitHub, Spotify, most modern APIs |
| **Basic auth** | Username + password encoded and sent in header | Older APIs |
| **OAuth 2.0** | A multi-step flow that grants a token | Google, Facebook login |

Today we will work with **API key in a query parameter** (OpenWeatherMap) because it is the most common pattern for beginners and the most visible — you can see it in the URL.

---

## Getting your OpenWeatherMap API key

OpenWeatherMap offers a free tier with 60 calls/minute and 1,000,000 calls/month — more than enough for learning.

**Steps to get your key:**
1. Go to [https://openweathermap.org](https://openweathermap.org) and click **Sign In / Sign Up**
2. Create a free account and verify your email
3. Go to **My Profile → API keys**
4. Copy the default key (it looks like: `a1b2c3d4e5f6a1b2c3d4e5f6a1b2c3d4`)
5. Paste it in the cell below

> **Note:** New API keys take up to 2 hours to activate after signup. If you see a 401 error right after creating your account, wait a little and try again.

---

## A critical rule: never hard-code your API key

**Never do this:**
```python
API_KEY = "a1b2c3d4e5f6a1b2c3d4e5f6a1b2c3d4"   # BAD
```

If you push this notebook to GitHub, share it with someone, or accidentally make it public, your key is exposed. Anyone who finds it can use your account.

**The right way for a notebook:** use `input()` to ask for the key at runtime. It is typed in but never stored in the file itself.

In [ ]:
# Enter your OpenWeatherMap API key here.
# It will be stored in the variable WEATHER_API_KEY for use in the cells below.
# It is NOT saved in the notebook file.

WEATHER_API_KEY = input("Paste your OpenWeatherMap API key and press Enter: ").strip()

if len(WEATHER_API_KEY) > 10:
    print(f"Key received ({len(WEATHER_API_KEY)} characters). Do not share this key.")
else:
    print("Key looks too short. Double-check and re-run this cell.")

In [ ]:
# Make your first authenticated API call
# OpenWeatherMap's current weather endpoint
# Documentation: https://openweathermap.org/api/current

BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "q":     "Mumbai",      # city name
    "appid": WEATHER_API_KEY,  # authentication — the API key
    "units": "metric"       # temperature in Celsius (default is Kelvin)
}

response = requests.get(BASE_URL, params=params)

print("Status code:", response.status_code)
print()
print("Full URL called (notice the key in the URL):")
print(" ", response.url)
print()

if response.status_code == 200:
    data = response.json()
    print("Top-level keys in the response:")
    for key in data.keys():
        print(f"  {key}")

In [ ]:
# Extract the useful fields from the weather response

if response.status_code == 200:
    data = response.json()

    city        = data["name"]
    country     = data["sys"]["country"]
    description = data["weather"][0]["description"]
    temp        = data["main"]["temp"]
    feels_like  = data["main"]["feels_like"]
    humidity    = data["main"]["humidity"]
    wind_speed  = data["wind"]["speed"]

    print(f"Weather in {city}, {country}")
    print("=" * 35)
    print(f"  Condition:   {description.title()}")
    print(f"  Temperature: {temp} C")
    print(f"  Feels like:  {feels_like} C")
    print(f"  Humidity:    {humidity}%")
    print(f"  Wind speed:  {wind_speed} m/s")
else:
    print(f"Could not fetch weather. Status: {response.status_code}")
    print(response.json())

In [ ]:
# Demonstrate what happens with a wrong API key
# This is intentionally broken to show the 401 response

bad_params = {
    "q":     "Delhi",
    "appid": "this_is_a_fake_key_12345",
    "units": "metric"
}

bad_response = requests.get(BASE_URL, params=bad_params)

print(f"Status code: {bad_response.status_code}")
print(f"Response body:")
print(json.dumps(bad_response.json(), indent=2))
print()
print("This is a 401 Unauthorized — the API rejected the fake key.")

---
### Exercise 2 — Authenticated API Calls

**Task:** Complete the function below. It should fetch weather data for a given city name and return a clean summary dictionary. It must handle the three most likely failure cases: invalid key (401), city not found (404), and any other error.

In [ ]:
def get_weather(city, api_key):
    """
    Fetches current weather for a city.
    Returns a summary dict on success, or None on failure.
    """
    params = {
        "q":     ___,          # <-- city name
        "appid": ___,          # <-- api key
        "units": "metric"
    }

    response = requests.get("https://api.openweathermap.org/data/2.5/weather", params=params)
    code = response.status_code

    if code == 200:
        d = response.json()
        return {
            "city":        d["name"],
            "country":     d[___]["country"],    # <-- which nested key?
            "temp_c":      d["main"]["temp"],
            "humidity":    d["main"][___],        # <-- which field?
            "description": d["weather"][0][___]  # <-- which field?
        }

    elif code == 401:
        print("Authentication failed. Check your API key.")
        return ___

    elif code == 404:
        print(f"City '{city}' not found.")
        return ___

    else:
        print(f"Error {code}: {response.json().get('message', 'Unknown error')}")
        return ___


# Test with a real city
result = get_weather("Bangalore", WEATHER_API_KEY)
if result:
    print(f"Weather in {result['city']}, {result['country']}:")
    print(f"  Temp: {result['temp_c']} C | Humidity: {result['humidity']}% | {result['description'].title()}")

print()

# Test with a fake city
result2 = get_weather("Fakecityxyz", WEATHER_API_KEY)
print(f"Fake city returned: {result2}")

<details>
<summary>Show solution</summary>

```python
def get_weather(city, api_key):
    params = {"q": city, "appid": api_key, "units": "metric"}
    response = requests.get("https://api.openweathermap.org/data/2.5/weather", params=params)
    code = response.status_code

    if code == 200:
        d = response.json()
        return {
            "city":        d["name"],
            "country":     d["sys"]["country"],
            "temp_c":      d["main"]["temp"],
            "humidity":    d["main"]["humidity"],
            "description": d["weather"][0]["description"]
        }
    elif code == 401:
        print("Authentication failed. Check your API key.")
        return None
    elif code == 404:
        print(f"City '{city}' not found.")
        return None
    else:
        print(f"Error {code}: {response.json().get('message', 'Unknown error')}")
        return None
```

</details>

---
# Part 3 — Header Management

## What are headers?

Headers are **metadata** — information about a request or response that travels alongside the actual data. Think of them as the envelope around a letter. The letter is the body (your JSON data). The envelope contains the address, the stamp, and the sender's details — that is the headers.

Headers are key-value pairs, like a dictionary. They appear in both requests (you send them) and responses (the server sends them back).

---

## Common request headers you will send

| Header | Purpose | Example value |
|---|---|---|
| `Authorization` | Carry an API key or token | `Bearer abc123` |
| `Content-Type` | Tell the server what format your body is in | `application/json` |
| `Accept` | Tell the server what format you want back | `application/json` |
| `User-Agent` | Identify your client (app name/version) | `LibraryApp/1.0` |
| `X-API-Key` | Another common way to send a key | `abc123` |

## Common response headers to read

| Header | What it tells you | Example value |
|---|---|---|
| `Content-Type` | Format of the response body | `application/json; charset=utf-8` |
| `X-RateLimit-Limit` | How many requests your key allows per window | `60` |
| `X-RateLimit-Remaining` | How many you have left in this window | `58` |
| `X-RateLimit-Reset` | When the limit resets (Unix timestamp) | `1717056000` |
| `Link` | Pagination URLs (next page, previous page) | `<url>; rel="next"` |
| `Retry-After` | How many seconds to wait before retrying (on 429) | `30` |

In [ ]:
# Reading response headers
# Let's see what headers come back from a real API call

response = requests.get(
    "https://restcountries.com/v3.1/name/india",
    params={"fields": "name,population"}
)

print("Response headers:")
print("=" * 50)
for key, value in response.headers.items():
    print(f"  {key:<35} {value}")

In [ ]:
# Sending custom headers with a request
# Use the 'headers' argument in requests.get() or requests.post()

# This is particularly important for APIs that use
# 'Authorization: Bearer <token>' authentication
# (e.g. GitHub, Spotify, most modern APIs)

custom_headers = {
    "Accept":     "application/json",
    "User-Agent": "LibraryAPISession/1.0"
}

response = requests.get(
    "https://httpbin.org/headers",   # HTTPBin echoes the headers it received
    headers=custom_headers
)

echo = response.json()
print("Headers the server received from us:")
for key, value in echo["headers"].items():
    print(f"  {key:<30} {value}")

In [ ]:
# Bearer token authentication — the most common modern pattern
# The key goes in the Authorization header, NOT in the URL
# This is more secure because it does not appear in server logs or browser history

# Demonstration with HTTPBin (it will echo the header back to us)
fake_token = "eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.example"

auth_headers = {
    "Authorization": f"Bearer {fake_token}",
    "Accept":        "application/json"
}

response = requests.get("https://httpbin.org/bearer", headers=auth_headers)

print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")
print()
print("Notice: the token is in the Authorization header, not in the URL.")
print(f"URL: {response.url}")

In [ ]:
# Reading rate limit headers from the GitHub API
# GitHub sends X-RateLimit headers on every response

response = requests.get(
    "https://api.github.com/repos/python/cpython",
    headers={"Accept": "application/vnd.github+json"}
)

print(f"Status: {response.status_code}")
print()
print("Rate limit headers from GitHub:")
rate_headers = [
    "X-RateLimit-Limit",
    "X-RateLimit-Remaining",
    "X-RateLimit-Used",
    "X-RateLimit-Reset"
]
for h in rate_headers:
    value = response.headers.get(h, "not present")
    print(f"  {h:<28} {value}")

print()
print("These headers tell you how many requests you have left before being rate-limited.")

---
### Exercise 3 — Working with Headers

**Task:** Complete the function below. It should:
1. Make a GET request to the OpenWeatherMap API with your key in the `params`
2. Also send a custom `User-Agent` header identifying your app as `"WeatherDashboard/1.0"`
3. Read the `Content-Type` header from the response and return it alongside the weather data

In [ ]:
def get_weather_with_headers(city, api_key):
    """
    Fetches weather and also returns the Content-Type response header.
    Returns a dict with 'city', 'temp_c', and 'content_type'.
    """
    params = {
        "q":     city,
        "appid": api_key,
        "units": "metric"
    }

    # Build the request headers dict
    req_headers = {
        "User-Agent": ___,     # <-- what string?
        "Accept":     "application/json"
    }

    response = requests.get(
        "https://api.openweathermap.org/data/2.5/weather",
        params=___,            # <-- which variable?
        headers=___            # <-- which variable?
    )

    if response.status_code == 200:
        d = response.json()
        return {
            "city":         d["name"],
            "temp_c":       d["main"]["temp"],
            "content_type": response.___["Content-Type"]   # <-- how to access headers?
        }
    return None


result = get_weather_with_headers("Chennai", WEATHER_API_KEY)
if result:
    print(f"City:          {result['city']}")
    print(f"Temperature:   {result['temp_c']} C")
    print(f"Content-Type:  {result['content_type']}")
    print()
    print(f"Correct (Content-Type contains 'json'): {'json' in result['content_type']}")

<details>
<summary>Show solution</summary>

```python
def get_weather_with_headers(city, api_key):
    params = {"q": city, "appid": api_key, "units": "metric"}
    req_headers = {"User-Agent": "WeatherDashboard/1.0", "Accept": "application/json"}
    response = requests.get(
        "https://api.openweathermap.org/data/2.5/weather",
        params=params,
        headers=req_headers
    )
    if response.status_code == 200:
        d = response.json()
        return {
            "city":         d["name"],
            "temp_c":       d["main"]["temp"],
            "content_type": response.headers["Content-Type"]
        }
    return None
```

</details>

---
# Part 4 — Pagination

## The problem

Imagine you ask GitHub: "Give me all the repositories in the Python organisation."

There are over 200 of them. If the API returned all 200 in one response, that is a large JSON payload. For an API used by thousands of developers simultaneously, sending giant responses for every request would be very slow and expensive.

So instead, the API says: **"I will give you 30 at a time. Ask me for the next 30 when you are ready."**

This is **pagination** — breaking a large dataset into pages.

---

## Two styles of pagination

**Style 1: Page-number pagination (offset-based)**

You request specific page numbers: `?page=1`, `?page=2`, `?page=3`, etc. Simple and predictable.

```
Request 1: ?page=1&per_page=30  →  items 1–30
Request 2: ?page=2&per_page=30  →  items 31–60
Request 3: ?page=3&per_page=30  →  items 61–90
...
```

**Style 2: Cursor/Link-header pagination**

The API tells you in the response headers exactly what URL to call next. More reliable for fast-changing datasets.

GitHub uses both. We will use page-number pagination today because it is the most common and the easiest to understand.

---

## The GitHub API

We will use the GitHub API to fetch repositories from the `python` organisation.

- Endpoint: `https://api.github.com/orgs/python/repos`
- Default: 30 repos per page
- Maximum: 100 repos per page (with `?per_page=100`)
- No auth needed for public data (but rate limited to 60 requests/hour without a token)
- Pagination info lives in the `Link` response header

In [ ]:
# First, let's fetch just page 1 and see what we get

url    = "https://api.github.com/orgs/python/repos"
params = {"per_page": 10, "page": 1}  # ask for 10 per page so we can see pagination clearly

headers = {"Accept": "application/vnd.github+json"}

response = requests.get(url, params=params, headers=headers)

print(f"Status: {response.status_code}")
print(f"Repos on this page: {len(response.json())}")
print()

# The Link header tells us about other pages
link_header = response.headers.get("Link", "not present")
print("Link header:")
print(" ", link_header)
print()
print("Rate limit remaining:", response.headers.get("X-RateLimit-Remaining", "N/A"))

### Reading the Link header

The `Link` header looks like this:

```
<https://api.github.com/orgs/python/repos?per_page=10&page=2>; rel="next",
<https://api.github.com/orgs/python/repos?per_page=10&page=22>; rel="last"
```

Each entry is a URL in angle brackets, followed by a `rel` (relationship) label:

| `rel` value | Meaning |
|---|---|
| `next` | URL for the next page |
| `prev` | URL for the previous page |
| `first` | URL for the first page |
| `last` | URL for the last page |

If there is no `next` link, you are on the last page.

In [ ]:
# A helper function to parse the Link header
# Extracts the URLs by their rel label

def parse_link_header(link_header_value):
    """
    Parses the Link header and returns a dict of {rel: url}.
    Example output: {'next': 'https://...', 'last': 'https://...'}
    """
    if not link_header_value:
        return {}

    links = {}
    for part in link_header_value.split(","):
        part = part.strip()
        # Each part looks like: <URL>; rel="name"
        if ";" not in part:
            continue
        url_part, rel_part = part.split(";", 1)
        url = url_part.strip().strip("<>")
        rel = rel_part.strip().replace('rel="', "").replace('"', "")
        links[rel] = url
    return links


# Test it on the Link header we got
links = parse_link_header(response.headers.get("Link", ""))
print("Parsed Link header:")
for rel, url in links.items():
    print(f"  {rel:<8} -> {url}")

print()
if "last" in links:
    # Extract the page number from the 'last' URL
    last_url   = links["last"]
    last_page  = int(last_url.split("page=")[-1].split("&")[0])
    print(f"Total pages available: {last_page}")

In [ ]:
# Now let's fetch multiple pages and collect all the results
# We will fetch the first 3 pages (30 repos) and stop
# In real use you would continue until there is no 'next' link

def fetch_repos_paginated(org, per_page=10, max_pages=3):
    """
    Fetches repositories from a GitHub organisation across multiple pages.
    Stops after max_pages pages or when there are no more pages.
    Returns a list of repo names.
    """
    url     = f"https://api.github.com/orgs/{org}/repos"
    headers = {"Accept": "application/vnd.github+json"}
    all_repos   = []
    current_page = 1

    while current_page <= max_pages:
        print(f"  Fetching page {current_page}...", end=" ")

        params   = {"per_page": per_page, "page": current_page}
        response = requests.get(url, params=params, headers=headers)

        if response.status_code != 200:
            print(f"Error {response.status_code} — stopping.")
            break

        page_data = response.json()
        if len(page_data) == 0:
            print("No more items to check")
            break
        all_repos.extend(page_data)
        print(f"got {len(page_data)} repos (total so far: {len(all_repos)})")

        # # Check if there is a next page
        # links = parse_link_header(response.headers.get("Link", ""))
        # if "next" not in links:
        #     print("  No 'next' link — this was the last page.")
        #     break

        current_page += 1
        time.sleep(0.5)    # be polite — small pause between requests

    return all_repos


print("Fetching Python organisation repositories:")
repos = fetch_repos_paginated(org="python", per_page=10, max_pages=30)
print()
print(f"Total repos fetched: {len(repos)}")
print()
print("First 5 repos:")
for repo in repos[:5]:
    stars = repo.get('stargazers_count', 0)
    print(f"  {repo['name']:<35} stars: {stars:>6,}")

Fetching Python organisation repositories:
  Fetching page 1... got 10 repos (total so far: 10)
  Fetching page 2... got 10 repos (total so far: 20)
  Fetching page 3... got 10 repos (total so far: 30)
  Fetching page 4... got 10 repos (total so far: 40)
  Fetching page 5... got 10 repos (total so far: 50)
  Fetching page 6... got 10 repos (total so far: 60)
  Fetching page 7... got 10 repos (total so far: 70)
  Fetching page 8... got 10 repos (total so far: 80)
  Fetching page 9... got 10 repos (total so far: 90)
  Fetching page 10... got 1 repos (total so far: 91)
  Fetching page 11... No more items to check

Total repos fetched: 91

First 5 repos:
  getpython3.com                      stars:     46
  community-starter-kit               stars:     49
  psf-docs                            stars:     22
  historic-python-materials           stars:     22
  psf-chef                            stars:     46


In [ ]:
# The complete pattern: keep going until there is no 'next' link
# This gets ALL repositories, however many pages that takes

def fetch_all_repos(org, per_page=30):
    """
    Fetches ALL repositories for a GitHub organisation.
    Follows the 'next' link until there are no more pages.
    """
    url     = f"https://api.github.com/orgs/{org}/repos"
    headers = {"Accept": "application/vnd.github+json"}
    all_repos    = []
    page         = 1

    while True:
        params   = {"per_page": per_page, "page": page}
        response = requests.get(url, params=params, headers=headers)

        if response.status_code != 200:
            break

        page_data = response.json()
        if not page_data:   # empty list — no more data
            break

        all_repos.extend(page_data)

        links = parse_link_header(response.headers.get("Link", ""))
        if "next" not in links:   # no next link — we are done
            break

        page += 1
        time.sleep(0.3)

    return all_repos


print("Fetching ALL Python org repos (this may take a moment)...")
all_repos = fetch_all_repos(org="python", per_page=30)

print(f"Total repos found: {len(all_repos)}")
print()

# Sort by stars and show the top 10
sorted_repos = sorted(all_repos, key=lambda r: r.get("stargazers_count", 0), reverse=True)
print(f"Top 10 most starred repos in the Python organisation:")
print(f"  {'Repo name':<40} {'Stars':>8}  {'Language'}")
print("  " + "-" * 65)
for repo in sorted_repos[:10]:
    name     = repo["name"]
    stars    = repo.get("stargazers_count", 0)
    language = repo.get("language") or "N/A"
    print(f"  {name:<40} {stars:>8,}  {language}")

---
### Exercise 4 — Pagination Logic

**Task:** Complete the function below. It should fetch repositories from a GitHub organisation page by page, but stop early as soon as it has collected at least `target_count` repos — instead of fetching everything.

This is a common real-world pattern: you do not always need all the data, just enough for your use case.

In [ ]:
def fetch_repos_until(org, target_count, per_page=10):
    """
    Fetches repos page by page, stopping once target_count is reached.
    Returns a list of repo name strings.
    """
    url      = f"https://api.github.com/orgs/{org}/repos"
    headers  = {"Accept": "application/vnd.github+json"}
    collected = []
    page     = 1

    while True:
        # Stop if we already have enough
        if len(collected) >=target_count:    # <-- stop condition
            break

        params   = {"per_page": per_page, "page":page}    # <-- which variable?
        response = requests.get(url, params=params, headers=headers)

        if response.status_code != 200:
            print(f"Error {response.status_code}")
            break

        page_data = response.json()

        if not page_data:    # empty page — no more data
            break

        for repo in page_data:
            collected.append(repo["name"])

        print(f"  Page {page}: fetched {len(page_data)}, total so far: {len(collected)}")

        # Check if there is a next page
        links = parse_link_header(response.headers.get("Link", ""))
        if "next" not in links:    # <-- what are we checking for?
            break

        page += 1   # <-- increment
        time.sleep(0.3)

    # Return only up to target_count (we may have gone slightly over)
    return collected[:___]    # <-- slice to target_count


results = fetch_repos_until(org="python", target_count=25, per_page=10)
print()
print(f"Repos collected: {len(results)}")
print(f"First 5: {results[:5]}")
print()
print(f"Correct (exactly 25 repos): {len(results) == 25}")

<details>
<summary>Show solution</summary>

```python
def fetch_repos_until(org, target_count, per_page=10):
    url       = f"https://api.github.com/orgs/{org}/repos"
    headers   = {"Accept": "application/vnd.github+json"}
    collected = []
    page      = 1

    while True:
        if len(collected) >= target_count:
            break

        params   = {"per_page": per_page, "page": page}
        response = requests.get(url, params=params, headers=headers)

        if response.status_code != 200:
            print(f"Error {response.status_code}")
            break

        page_data = response.json()
        if not page_data:
            break

        for repo in page_data:
            collected.append(repo["name"])

        print(f"  Page {page}: fetched {len(page_data)}, total so far: {len(collected)}")

        links = parse_link_header(response.headers.get("Link", ""))
        if "next" not in links:
            break

        page += 1
        time.sleep(0.3)

    return collected[:target_count]
```

</details>

---
## Part 5 — Putting It All Together

We now have all four tools. Let's build something that uses all of them:

A small **weather dashboard** that:
- Takes a list of Indian cities
- Fetches weather for each one using the authenticated OpenWeatherMap API
- Sends proper headers
- Handles errors per city (one failure should not stop the whole run)
- Reads a rate-limit-related header from the final response
- Prints a sorted summary table

In [ ]:
def build_weather_dashboard(cities, api_key):
    """
    Fetches weather for a list of cities and returns a summary.
    Uses auth, headers, status code handling, and a small delay between calls.
    """
    results  = []
    errors   = []

    req_headers = {
        "Accept":     "application/json",
        "User-Agent": "WeatherDashboard/1.0"
    }

    for city in cities:
        params = {
            "q":     city,
            "appid": api_key,
            "units": "metric"
        }

        try:
            response = requests.get(
                "https://api.openweathermap.org/data/2.5/weather",
                params=params,
                headers=req_headers,
                timeout=8
            )

            if response.status_code == 200:
                d = response.json()
                results.append({
                    "city":        d["name"],
                    "country":     d["sys"]["country"],
                    "temp_c":      d["main"]["temp"],
                    "feels_like":  d["main"]["feels_like"],
                    "humidity":    d["main"]["humidity"],
                    "description": d["weather"][0]["description"].title()
                })

            elif response.status_code == 401:
                print(f"  AUTH ERROR: Invalid API key — stopping all requests.")
                break   # no point continuing if the key is bad

            elif response.status_code == 404:
                print(f"  SKIP: '{city}' not found")
                errors.append(city)

            elif response.status_code == 429:
                retry_after = response.headers.get("Retry-After", "unknown")
                print(f"  RATE LIMITED: Wait {retry_after}s before continuing.")
                break

            else:
                print(f"  ERROR {response.status_code} for '{city}'")
                errors.append(city)

        except requests.exceptions.Timeout:
            print(f"  TIMEOUT for '{city}'")
            errors.append(city)

        time.sleep(0.4)  # stay well within the free tier rate limit

    return results, errors


# Run the dashboard
cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata",
          "Hyderabad", "Pune", "Ahmedabad", "Notacity"]

print("Fetching weather data...")
results, errors = build_weather_dashboard(cities, WEATHER_API_KEY)

print()
print("WEATHER DASHBOARD")
print("=" * 80)
print(f"  {'City':<15} {'Country':<10} {'Temp':>6}  {'Feels':<9} {'Humidity':>10}  {'Condition'}")
print("  " + "-" * 75)

# Sort by temperature, hottest first
for r in sorted(results, key=lambda x: x["temp_c"], reverse=True):
    print(f"  {r['city']:<15} {r['country']:<10} {r['temp_c']:>5.1f}C  {r['feels_like']:<7.1f}C  {r['humidity']:>8}%  {r['description']}")

print()
print(f"Successful: {len(results)} cities")
if errors:
    print(f"Skipped:    {errors}")

---
## Final Exercise — End-to-End Challenge

This exercise brings together all four topics from today.

**Scenario:** You are building a data collection tool for a tech report. The tool needs to:

1. Fetch repositories from the `microsoft` GitHub organisation — at least 40 repos, paginated
2. Filter to keep only repos that have more than 1000 stars
3. For each kept repo, extract: `name`, `stars`, `language`, `description`
4. Handle any non-200 status code gracefully with a clear message
5. Print the final results sorted by stars (most starred first)

Complete the blanks in the three cells below.

In [ ]:
# Step 1: Fetch at least 40 repos from the microsoft org, paginated

def fetch_org_repos(org, min_count, per_page=20):
    """
    Fetches at least min_count repos from a GitHub org.
    Returns a list of raw repo objects.
    """
    url      = f"https://api.github.com/orgs/{org}/repos"
    headers  = {"Accept": "application/vnd.github+json"}
    collected = []
    page     = 1

    while len(collected) <min_count:         # <-- stop when we have enough
        params   = {"per_page": per_page, "page": page, "sort": "stars"}
        response = requests.get(url, params=params, headers=headers)

        if response.status_code != 200:         # <-- how to get the status code?
            print(f"Error {response.status_code} on page {page}")
            break

        page_data = response.json()
        if not page_data:
            break

        collected.extend(page_data)
        print(f"  Page {page}: {len(page_data)} repos fetched, total: {len(collected)}")

        links = parse_link_header(response.headers.get("Link", ""))
        if "next" not in links:
            break

        page += 1
        time.sleep(0.3)

    return collected


print("Fetching Microsoft repos...")
raw_repos = fetch_org_repos("microsoft", min_count=40, per_page=20)
print(f"Total fetched: {len(raw_repos)}")

In [ ]:
# Step 2 & 3: Filter and extract fields

popular_repos = []

for repo in raw_repos:
    stars = repo.get("stargazers_count", 0)

    if stars > 1000:                    # <-- filter threshold
        popular_repos.append({
            "name":        repo["name"],
            "stars":       repo["stargazers_count"],        # <-- which field?
            "language":    repo.get("language") or "N/A",
            "description": (repo.get("description") or "")[:60]   # truncate long descriptions
        })

print(f"Repos with more than 1000 stars: {len(popular_repos)}")

In [ ]:
# Step 4 & 5: Sort and display

sorted_repos = sorted(popular_repos, key=lambda r: r["stars"], reverse=True)  # <-- sort by stars

print("TOP MICROSOFT REPOSITORIES (>1000 stars)")
print("=" * 80)
print(f"  {'Name':<35} {'Stars':>8}  {'Language':<15}  Description")
print("  " + "-" * 78)

for repo in sorted_repos:
    print(f"  {repo['name']:<35} {repo['stars']:>8,}  {repo['language']:<15}  {repo['description']}")

print()
print(f"Correct (at least 5 popular repos found): {len(sorted_repos) >= 5}")
print(f"Correct (sorted descending by stars):     {all(sorted_repos[i]['stars'] >= sorted_repos[i+1]['stars'] for i in range(len(sorted_repos)-1))}")

---
## Session Summary

### 1. Status Codes
- Always check the status code before using the response body
- `response.ok` is a quick True/False for 2xx codes
- `response.status_code` gives you the exact number for granular handling
- `response.raise_for_status()` throws an exception on 4xx/5xx — useful with try/except
- Always add a `timeout=` to prevent your program hanging indefinitely
- **Rule of thumb:** 4xx = you made an error, 5xx = the server made an error

### 2. Authentication
- Most real-world APIs require a key — it controls rate limits, billing, and access
- Never hard-code keys in your code files — use `input()` in notebooks, environment variables in scripts
- **Query param method:** `?appid=yourkey` (OpenWeatherMap style)
- **Header method:** `Authorization: Bearer yourtoken` (GitHub, Spotify style) — more secure
- A 401 means your key is wrong or missing. A 403 means your key works but lacks permission.

### 3. Header Management
- Headers travel alongside requests and responses as key-value metadata
- **Sending headers:** `requests.get(url, headers={"User-Agent": "MyApp/1.0"})`
- **Reading headers:** `response.headers["Content-Type"]`
- Rate limit headers (`X-RateLimit-Remaining`, `Retry-After`) tell you when to slow down
- The `Link` header carries pagination URLs for next/previous/last pages

### 4. Pagination
- APIs do not return all results at once — they return a page at a time (usually 30 or 100 items)
- Use `?page=N&per_page=N` query parameters to navigate pages
- Check the `Link` response header for the `next` URL — when it is absent, you are on the last page
- Always add a small `time.sleep()` between requests to avoid hitting rate limits
- Pattern: `while True` loop, break when `"next" not in links` or when you have enough data

---

### Quick reference — patterns used today

```python
import requests, time

# Status code handling
r = requests.get(url, params=params, timeout=10)
if r.status_code == 200: ...
elif r.status_code == 404: ...
elif r.status_code == 429: time.sleep(int(r.headers.get("Retry-After", 30)))

# Authentication — key in params
params = {"q": city, "appid": MY_KEY, "units": "metric"}

# Authentication — key in header (Bearer token)
headers = {"Authorization": f"Bearer {MY_TOKEN}"}

# Reading response headers
content_type     = r.headers["Content-Type"]
rate_limit_left  = r.headers.get("X-RateLimit-Remaining")

# Pagination loop
page = 1
while True:
    r = requests.get(url, params={"page": page, "per_page": 30})
    data = r.json()
    if not data: break
    all_results.extend(data)
    links = parse_link_header(r.headers.get("Link", ""))
    if "next" not in links: break
    page += 1
    time.sleep(0.3)
```